# 🧠 AI Logic Engine: Main Inference Console

ဤ Notebook သည် Firestore အတွင်းရှိ **Knowledge Graph** ကို အခြေခံ၍ Concept များအကြား Logical Connection ရှာရန်နှင့် အဖြေထုတ်ရန် အသုံးပြုသည်။

---

In [ ]:
# @title 🛠️ Step 1: Initialize Inference Memory
!pip install -q firebase-admin

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore
import re

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Inference Memory Connected: {DATABASE_ID}")

In [ ]:
# @title ⚙️ Step 2: Symbolic Inference Core

class KnowledgeInferenceEngine:
    def __init__(self, db):
        self.db = db
        self.cache = {}

    def clean_id(self, text):
        # Must strictly match the Ingestion Notebook normalization
        return re.sub(r'[^a-z0-9]', '_', text.lower().strip())

    def get_node(self, node_id):
        node_key = self.clean_id(node_id)
        if not node_key: return None
        if node_key in self.cache: return self.cache[node_key]
        
        doc = self.db.collection('nodes').document(node_key).get()
        if doc.exists:
            data = doc.to_dict()
            self.cache[node_key] = data
            return data
        return None

    def find_logical_proof(self, start, target, max_depth=6):
        """Symbolic Pathfinding using Transitive Logic Chains."""
        start_id = self.clean_id(start)
        target_id = self.clean_id(target)
        
        # Queue stores: (current_id, path_history, certainty_score)
        queue = [(start_id, [], 1.0)]
        visited = {start_id}
        
        while queue:
            curr_id, path, score = queue.pop(0)
            
            if curr_id == target_id:
                return path, score
            
            if len(path) >= max_depth: continue
            
            node = self.get_node(curr_id)
            if not node: continue
            
            # 1. Transitive Inheritance (is_a / Groups)
            for parent_id in node.get('groups', []):
                if parent_id not in visited:
                    visited.add(parent_id)
                    step_text = f"{node.get('name', curr_id)} is a type of {parent_id}"
                    queue.append((parent_id, path + [step_text], score * 0.99))
            
            # 2. Sequential Relations
            for rel in node.get('relations', []):
                t_id = rel['targetId']
                if t_id not in visited:
                    visited.add(t_id)
                    link_text = rel.get('sentence', f"{node.get('name', curr_id)} {rel['verb']} {t_id}")
                    queue.append((t_id, path + [link_text], score * 0.95))
                    
        return None, 0

engine = KnowledgeInferenceEngine(db)
print("✅ Inference Core Ready.")

In [ ]:
# @title 🔍 Step 3: Global Logic Query Console
START_CONCEPT = "Socrates" # @param {type:"string"}
TARGET_CONCEPT = "Philosophy" # @param {type:"string"}

print(f"🧠 Reasoning Path: '{START_CONCEPT}' -> '{TARGET_CONCEPT}'...\n")
proof, confidence = engine.find_logical_proof(START_CONCEPT, TARGET_CONCEPT)

if proof:
    print(f"✅ Logic Proof Established (Certainty: {confidence*100:.1f}%) ")
    for i, step in enumerate(proof):
        print(f"  Step {i+1}: {step}")
else:
    print("❌ No logical chain found in the current Memory Graph.")